In [1]:
import pandas as pd
from glob import glob
from os.path import basename

CONVERTERS = {"actually_used": lambda a: int(a) == 1, "num_ops": int, "num_bytes": lambda x: int(x) * 8, "max_rss": lambda r: int(r) * 1024, "expr": str}

In [2]:
csvs = glob("diexpr-bench/RelWithDebInfo/*")

In [3]:
df = pd.concat([pd.read_csv(fname, sep=':', names=CONVERTERS.keys(), converters=CONVERTERS).assign(compilation_id=basename(fname)) for fname in csvs], ignore_index=True)

In [4]:
df = df.infer_objects()

In [5]:
df.dtypes

actually_used       bool
num_ops            int64
num_bytes          int64
max_rss            int64
expr              object
compilation_id    object
dtype: object

In [6]:
df

,actually_used,num_ops,num_bytes,max_rss,expr,compilation_id
0,True,2,24,197922816,"!DIExpression(DW_OP_plus_uconst, 24, DW_OP_sta...",0eb4a07e14
1,False,2,40,197922816,"!DIExpression(DW_OP_LLVM_arg, 0, DW_OP_LLVM_fr...",0eb4a07e14
2,True,1,24,197922816,"!DIExpression(DW_OP_LLVM_fragment, 0, 80)",0eb4a07e14
3,True,2,24,197922816,"!DIExpression(DW_OP_plus_uconst, 72, DW_OP_sta...",0eb4a07e14
4,False,4,48,197922816,"!DIExpression(DW_OP_LLVM_arg, 0, DW_OP_constu,...",0eb4a07e14
...,...,...,...,...,...,...
1536291,True,1,24,202772480,"!DIExpression(DW_OP_LLVM_fragment, 64, 64)",b6dd03d455
1536292,True,2,24,202772480,"!DIExpression(DW_OP_plus_uconst, 72, DW_OP_deref)",b6dd03d455
1536293,False,3,40,202772480,"!DIExpression(DW_OP_LLVM_arg, 0, DW_OP_plus_uc...",b6dd03d455
1536294,False,3,40,202772480,"!DIExpression(DW_OP_LLVM_arg, 0, DW_OP_plus_uc...",b6dd03d455


In [7]:
df.describe()

,num_ops,num_bytes,max_rss
count,1.536296e+06,1.536296e+06,1.536296e+06
mean,4.337365e+00,6.026321e+01,4.329513e+08
std,4.933949e+00,6.390023e+01,2.140792e+08
min,0.000000e+00,0.000000e+00,7.893402e+07
25%,2.000000e+00,2.400000e+01,2.774426e+08
50%,3.000000e+00,4.000000e+01,3.841352e+08
75%,4.000000e+00,6.400000e+01,5.305590e+08
max,9.800000e+01,1.192000e+03,1.480258e+09


In [8]:
f"{df.actually_used.mean():.2%}"

'56.24%'

In [9]:
used_bytes = df.loc[df.actually_used == True].num_bytes.sum()

In [10]:
unused_bytes = df.loc[df.actually_used == False].num_bytes.sum()

In [11]:
f"{used_bytes / (used_bytes + unused_bytes):.2%}"

'59.94%'

In [12]:
unused_bytes / len(csvs)

7787.47417051659

In [17]:
df.groupby(['compilation_id','max_rss']).pipe(lambda g: g.num_bytes.sum() / g.max_rss.first()).describe()

count    4473.000000
mean        0.000057
std         0.000072
min         0.000000
25%         0.000011
50%         0.000034
75%         0.000076
max         0.001124
dtype: float64

In [14]:
# DIOp[]size / uint64_t[]size
df['diop_cost_ratio'] = ((df.num_ops * 16) / (df.num_bytes))

In [16]:
df['diop_cost_ratio'].describe()

count    1.531823e+06
mean     1.112958e+00
std      2.375864e-01
min      6.666667e-01
25%      1.000000e+00
50%      1.200000e+00
75%      1.333333e+00
max      2.000000e+00
Name: diop_cost_ratio, dtype: float64

In [15]:
df.sort_values(by=['diop_cost_ratio']).head()

,actually_used,num_ops,num_bytes,max_rss,expr,compilation_id,diop_cost_ratio
846379,True,1,24,624836608,"!DIExpression(DW_OP_LLVM_fragment, 512, 64)",8d0e77cb8b,0.666667
915141,True,1,24,525606912,"!DIExpression(DW_OP_LLVM_fragment, 576, 160)",06f9620534,0.666667
915143,False,1,24,525606912,"!DIExpression(DW_OP_LLVM_fragment, 192, 192)",06f9620534,0.666667
915144,True,1,24,525606912,"!DIExpression(DW_OP_LLVM_fragment, 0, 224)",06f9620534,0.666667
1284721,True,1,24,299380736,"!DIExpression(DW_OP_LLVM_fragment, 64, 32)",a211ebfce7,0.666667


In [33]:
df[~df['expr'].str.contains('DW_OP_LLVM_fragment')]['diop_cost_ratio'].describe()

count    1.011389e+06
mean     1.241406e+00
std      1.570306e-01
min      6.666667e-01
25%      1.200000e+00
50%      1.285714e+00
75%      1.333333e+00
max      2.000000e+00
Name: diop_cost_ratio, dtype: float64

In [35]:
df[~df['expr'].str.contains('DW_OP_LLVM_fragment')].sort_values(by=['diop_cost_ratio']).head()

,actually_used,num_ops,num_bytes,max_rss,expr,compilation_id,diop_cost_ratio
1014364,True,2,48,229965824,"!DIExpression(DW_OP_LLVM_convert, 64, DW_ATE_u...",760a6b03c0,0.666667
171244,True,2,48,195219456,"!DIExpression(DW_OP_LLVM_convert, 32, DW_ATE_u...",f72292c019,0.666667
38803,True,2,48,144519168,"!DIExpression(DW_OP_LLVM_convert, 32, DW_ATE_u...",e701a99f0e,0.666667
755801,True,2,48,190238720,"!DIExpression(DW_OP_LLVM_convert, 32, DW_ATE_u...",411d8e0bb2,0.666667
90173,True,2,48,155353088,"!DIExpression(DW_OP_LLVM_convert, 32, DW_ATE_u...",f22caab5f1,0.666667
